In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import evaluate
from features import FEATURES, TARGET, load_rated, make_splits

rated = load_rated()
train, valid, test = make_splits(rated)

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:<6} {len(part):>6,} businesses   fail rate {part[TARGET].mean():.2%}")

train  42,834 businesses   fail rate 5.61%
valid  14,278 businesses   fail rate 5.61%
test   14,278 businesses   fail rate 5.61%


In [2]:
rng = np.random.default_rng(0)
results = [evaluate("Random order", valid, rng.random(len(valid)))]
print(pd.DataFrame(results).round(3).to_string(index=False))

       model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
Random order       0.215            0.067            0.057             0.489


In [3]:
# Learn each business type's fail rate from the TRAINING set only
type_rates = train.groupby("BusinessType")[TARGET].mean()

# Types never seen in training fall back to the overall training fail rate
type_score = valid["BusinessType"].map(type_rates).fillna(train[TARGET].mean())

results.append(evaluate("Business type only", valid, type_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

             model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
      Random order       0.215            0.067            0.057             0.489
Business type only       0.324            0.078            0.080             0.639


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from features import CATEGORICAL, NUMERIC

SKEWED = ["name_count", "pop_density"]                   # long right tails: log first
OTHER_NUMERIC = [c for c in NUMERIC if c not in SKEWED]

preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("log_scaled", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), SKEWED),
    ("scaled", StandardScaler(), OTHER_NUMERIC),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000)),
])
logreg.fit(train[FEATURES], train[TARGET])

lr_score = logreg.predict_proba(valid[FEATURES])[:, 1]
results.append(evaluate("Logistic regression", valid, lr_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

              model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
       Random order       0.215            0.067            0.057             0.489
 Business type only       0.324            0.078            0.080             0.639
Logistic regression       0.374            0.115            0.151             0.749


In [5]:
import lightgbm as lgb

from features import CATEGORICAL


def lgb_frame(data, reference):
    """Feature table for LightGBM: categories stored as pandas 'category', using the
    category list from the reference (training) data so codes match across splits."""
    X = data[FEATURES].copy()
    for col in CATEGORICAL:
        X[col] = pd.Categorical(X[col], categories=sorted(reference[col].dropna().unique()))
    return X


X_train = lgb_frame(train, train)
X_valid = lgb_frame(valid, train)

lgbm = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
lgbm.fit(X_train, train[TARGET])

lgbm_score = lgbm.predict_proba(X_valid)[:, 1]
results.append(evaluate("LightGBM (default settings)", valid, lgbm_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

                      model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
               Random order       0.215            0.067            0.057             0.489
         Business type only       0.324            0.078            0.080             0.639
        Logistic regression       0.374            0.115            0.151             0.749
LightGBM (default settings)       0.397            0.127            0.152             0.751
